In [1]:
print("hi")

hi


```mermaid
graph LR
    A[CineBot Agent] -->|MCPAdapter| B[(MCP Server)]
    B --> C[check_showtimes]
    B --> D[cancel_booking]
    B --> E[get_seat_map]
```

In [3]:
import warnings,os
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
assert os.environ.get("OPENAI_API_KEY") is not None, "OPENAI_API_KEY is not set in the environment variables."

In [6]:
import langchain.mcp

/var/folders/gd/gpz62qhd2k59xc_wfb3qp1f80000gn/T/ipykernel_62893/1520507502.py:1: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  import langchain.mcp


# MCP Adapter in LangChain

In [ ]:
import langchain.mcp


In [17]:
%%writefile cinebot_mcp_server.py
from fastmcp import FastMCP
from mcp.types import ToolAnnotations

mcp = FastMCP("CineBot")

@mcp.tool()
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found.")

@mcp.tool(annotations=ToolAnnotations(destructiveHint=True))
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."

@mcp.tool()
def get_seat_map(movie_title: str) -> dict:
    """Get the seat map for a movie -- returns structured data, not just text."""
    return {"movie": movie_title, "available_rows": ["A", "B", "C"], "sold_out_rows": ["D"]}

if __name__ == "__main__":
    mcp.run()


Overwriting cinebot_mcp_server.py


In [8]:
from pathlib import Path
from langchain.mcp import MCPAdapter
from fastmcp.client.transports import PythonStdioTransport

In [19]:
cinebot_mcp_transport = PythonStdioTransport(
    script_path=Path("/Users/mayank/Github Repos/Agentic 3.0/Live-Class-2026/Weekend 14 - Oct 3 & Oct 4/MCP in Langchain/cinebot_mcp_server.py"),
    log_file=Path("cinebot_mcp_server.log")
)

async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    # Now you can use the mcp_adapter to interact with the MCP server
    tools=await mcp_adapter.list_tools()
    print("Available tools:", [t.name for t in tools])

Available tools: ['check_showtimes', 'cancel_booking', 'get_seat_map']
